In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

In [40]:
df_sales = pd.read_csv('../data/raw/sales.csv')
df_products = pd.read_csv('../data/raw/products.csv')
df_customers = pd.read_csv('../data/raw/customers.csv')

# Unindo vendas e clientes
df_unificado = pd.merge(df_sales, df_customers, on='Customer_ID')

# Unindo o resultado com os produtos
df_dataset = pd.merge(df_unificado, df_products, on='Product_ID')

# 1. Tratamento dos Valores Ausentes

In [41]:
df_dataset['Coupon_Code'] = df_dataset['Coupon_Code'].fillna('Sem_Cupom')
df_dataset['Review_Text'] = df_dataset['Review_Text'].fillna('Sem_Avaliacao')


In [42]:
# Força o preenchimento dos nulos (vazios reais) com o texto
df_dataset['Rating'] = df_dataset['Rating'].fillna('Sem_Avaliacao')

# Transforma tudo em texto para garantir que o One-Hot Encoding vai funcionar depois
df_dataset['Rating'] = df_dataset['Rating'].astype(str)

In [44]:
# Visualizando a quantidade e porcentagem de valores ausentes por coluna
ausentes = df_dataset.isnull().sum()
ausentes_pct = (ausentes / len(df_dataset) * 100).round(2)

tabela_ausentes = pd.DataFrame({"qtd_ausentes": ausentes, "pct_ausentes": ausentes_pct})
tabela_ausentes = tabela_ausentes[tabela_ausentes["qtd_ausentes"] > 0].sort_values("qtd_ausentes", ascending=False)
tabela_ausentes

,qtd_ausentes,pct_ausentes


# 2. Tratamento de Outliers
### DECISÃO: Manter todos os outliers. 
### Justificativa: Representam comportamentos reais de negócio (compras B2B, frete grátis, etc).
### Portanto, nenhuma linha será excluída nesta etapa.

# 3. Tratamento de Multicolinearidade (Removendo a redundância)

In [45]:
# Como Total_Amount e Selling_Price têm correlação de 0.88, removemos o Selling_Price
if 'Selling_Price' in df_dataset.columns:
    df_dataset = df_dataset.drop(columns=['Selling_Price'])
    print("\nColuna 'Selling_Price' removida com sucesso para evitar multicolinearidade.")


Coluna 'Selling_Price' removida com sucesso para evitar multicolinearidade.


# 4. Codificação de Variáveis Categóricas (One-Hot Encoding)

In [46]:
colunas_categoricas = ['Payment_Mode', 'Category', 'Gender', 'Rating'] 
df_dataset = pd.get_dummies(df_dataset, columns=colunas_categoricas, drop_first=True)

# 5. Padronização (Deixando os valores numéricos na mesma escala)

In [47]:
scaler = StandardScaler()
colunas_para_escalar = ['Total_Amount', 'Quantity', 'Shipping_Cost', 'Customer_Age']
df_dataset[colunas_para_escalar] = scaler.fit_transform(df_dataset[colunas_para_escalar])

# 6. Salvando o dataset pronto para o Machine Learning

In [48]:
df_dataset.to_csv('../data/processed/dataset_clean.csv', index=False)
print("\nPré-processamento concluído! Dataset salvo em data/processed/dataset_clean.csv")


Pré-processamento concluído! Dataset salvo em data/processed/dataset_clean.csv
